In [85]:
import pandas as pd
import numpy as np

df = pd.read_csv("cleaned-2.csv")

df.head()

,Unnamed: 0,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,business_or_commercial,loan_amount,rate_of_interest,...,credit_type,Credit_Score,LTV,Status,dtir1,loan_to_income,loan_to_collateral,interest_burden,total_cost_proxy,risk_score
0,0,cf,Sex Not Available,nopre,type1,p1,l1,nob/c,116500,3.99,...,EXP,758,98.728814,1,45.0,66.916,0.987,464835.0,467526.13,-16.8
1,1,cf,Male,nopre,type2,p1,l1,b/c,206500,3.99,...,EQUI,552,75.297619,1,40.0,41.458,0.494,823935.0,826626.13,104.8
2,2,cf,Male,pre,type1,p1,l1,nob/c,406500,4.56,...,EXP,834,80.019685,0,46.0,42.875,0.800,1853640.0,1854235.00,-62.0
3,3,cf,Male,nopre,type1,p4,l1,nob/c,456500,4.25,...,EXP,587,69.376900,0,42.0,38.423,0.694,1940125.0,1942816.13,84.6
4,4,cf,Joint,pre,type1,p1,l1,nob/c,696500,4.00,...,CRIF,602,91.886544,0,39.0,66.708,0.919,2786000.0,2786000.00,74.4


In [86]:
df = pd.get_dummies(df, drop_first=True)

y = df['Status']
X = df.drop(columns=['Status'])

In [87]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X = pd.DataFrame(X_scaled, columns=X.columns)

In [88]:
#FORWARD FEATURE SELECTION
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report,recall_score , f1_score

columns=['Status']

features = list(X.columns)
forward_selected = []
best_score = 0

while features:
    scores = []
    for f in features:
        temp = forward_selected + [f]
        X_train, X_test, y_train, y_test = train_test_split(
            X[temp], y, test_size=0.2, random_state=42)

        model1 = LogisticRegression(max_iter=3000)
        model1.fit(X_train, y_train)
        pred = model1.predict(X_test)
        score = recall_score(y_test, pred)
        scores.append((score, f))

    scores.sort(reverse=True)
    best, best_feature = scores[0]

    if best > best_score:
        forward_selected.append(best_feature)
        features.remove(best_feature)
        best_score = best
        print(f"Added: {best_feature} | Accuracy: {best}")
    else:
        break

print("Final Selected Features:\n", forward_selected)

Added: credit_type_EQUI | Accuracy: 0.35
Added: lump_sum_payment_not_lpsm | Accuracy: 0.375
Added: loan_to_income | Accuracy: 0.4
Final Selected Features:
 ['credit_type_EQUI', 'lump_sum_payment_not_lpsm', 'loan_to_income']


In [89]:
X = df[forward_selected]
y = df['Status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model2 = LogisticRegression(max_iter=1000)
model2.fit(X_train, y_train)

y_pred = model2.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy : 0.8311258278145696
Precision: 0.9142857142857143
Recall   : 0.4
F1-score : 0.5565217391304348

Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.99      0.90       222
           1       0.91      0.40      0.56        80

    accuracy                           0.83       302
   macro avg       0.87      0.69      0.73       302
weighted avg       0.85      0.83      0.81       302



In [90]:
#BACKWARD FEATURE SELECTION 
features = list(X.columns)
best_score = 0

while len(features) > 5:
    scores = []

    for f in features:
        temp = features.copy()
        temp.remove(f)

        X_train, X_test, y_train, y_test = train_test_split(
            X[temp], y, test_size=0.2, random_state=42
        )

        model3 = LogisticRegression(max_iter=3000)
        model3.fit(X_train, y_train)

        pred = model3.predict(X_test)
        score = recall_score(y_test, pred)

        scores.append((score, f))

    scores.sort(reverse=True)
    best, remove_feature = scores[0]

    if best > best_score - 0.01:
        features.remove(remove_feature)
        best_score = best
        print(f"Removed: {remove_feature} | Accuracy: {best}")
    else:
        break

print("Final Selected Features:\n", features)
backward_selected= features

Final Selected Features:
 ['credit_type_EQUI', 'lump_sum_payment_not_lpsm', 'loan_to_income']


In [91]:
X = df[backward_selected]
y = df['Status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model4 = LogisticRegression(max_iter=1000)
model4.fit(X_train, y_train)

y_pred = model4.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy : 0.8311258278145696
Precision: 0.9142857142857143
Recall   : 0.4
F1-score : 0.5565217391304348

Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.99      0.90       222
           1       0.91      0.40      0.56        80

    accuracy                           0.83       302
   macro avg       0.87      0.69      0.73       302
weighted avg       0.85      0.83      0.81       302



In [92]:
#PCA 
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
pca = PCA(n_components=10)
X_train_pca = pca.fit_transform(X_train)  
X_test_pca = pca.transform(X_test)    
model5 = LogisticRegression(max_iter=1000)
model5.fit(X_train_pca, y_train)
y_pred = model5.predict(X_test_pca)
y_pred = model5.predict(X_test_pca)


In [93]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)


Accuracy: 0.847682119205298
Precision: 0.9047619047619048
Recall: 0.475
F1-score: 0.6229508196721312

Classification Report:

              precision    recall  f1-score   support

           0       0.84      0.98      0.90       222
           1       0.90      0.47      0.62        80

    accuracy                           0.85       302
   macro avg       0.87      0.73      0.76       302
weighted avg       0.86      0.85      0.83       302

Confusion Matrix:
 [[218   4]
 [ 42  38]]


In [94]:
"here the PCA featur eselection gives the best recall and f1 thus avoiding false neagtives  therefore we will use that particular model as well as we can use backward/forward any model for future predictions "

'here the PCA featur eselection gives the best recall and f1 thus avoiding false neagtives  therefore we will use that particular model as well as we can use backward/forward any model for future predictions '

In [95]:
import pickle 
with open("model5.pkl", "wb") as f:
    pickle.dump(model5, f)
with open("model4.pkl","wb")as f:
    pickle.dump(model4,f)
print("Model saved successfully")

Model saved successfully
